# Atividade Prática 1 - Cálculo Numérico

**Nomes:** [Inserir nome dos integrantes]
**Divisão de tarefas:** [Inserir divisão de tarefas]



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import imageio
import os
from IPython.display import Image, display

# Funções de erro
def erro_absoluto(x, x_aprox):
    '''Calcula o erro absoluto.'''
    return np.abs(x - x_aprox)

def erro_relativo(x, x_aprox):
    '''Calcula o erro relativo.'''
    return np.abs(x - x_aprox) / np.abs(x)

def erro_percentual(x, x_aprox):
    '''Calcula o erro percentual.'''
    return 100 * erro_relativo(x, x_aprox)

# Função de truncamento
def truncar(x, n):
    '''Trunca o valor x para n casas decimais sem arredondar.'''
    fator = 10**n
    return np.trunc(fator * x) / fator

pd.set_option('display.float_format', lambda x: '%.10g' % x)



## Parte A – Truncamento e arredondamento


In [ ]:
# Valores
x_vals = [3.14159265, 98.76543, 0.00498765]
casas = [0, 1, 2, 3, 4]

resultados_A = []

for x in x_vals:
    for n in casas:
        # Truncamento
        x_trunc = truncar(x, n)
        ea_trunc = erro_absoluto(x, x_trunc)
        er_trunc = erro_relativo(x, x_trunc)
        ep_trunc = erro_percentual(x, x_trunc)
        
        # Arredondamento
        x_arred = round(x, n)
        ea_arred = erro_absoluto(x, x_arred)
        er_arred = erro_relativo(x, x_arred)
        ep_arred = erro_percentual(x, x_arred)
        
        resultados_A.append({
            'Valor (x)': x, 'n (casas)': n, 'Tipo': 'Truncamento',
            'Aprox': x_trunc, 'Ea': ea_trunc, 'Er': er_trunc, 'E%': ep_trunc
        })
        resultados_A.append({
            'Valor (x)': x, 'n (casas)': n, 'Tipo': 'Arredondamento',
            'Aprox': x_arred, 'Ea': ea_arred, 'Er': er_arred, 'E%': ep_arred
        })

df_A = pd.DataFrame(resultados_A)
display(df_A)

plt.figure(figsize=(12, 5))
for i, x in enumerate(x_vals):
    plt.subplot(1, 3, i+1)
    df_x = df_A[df_A['Valor (x)'] == x]
    
    trunc_data = df_x[df_x['Tipo'] == 'Truncamento']
    arred_data = df_x[df_x['Tipo'] == 'Arredondamento']
    
    plt.plot(trunc_data['n (casas)'], trunc_data['Ea'], 'o-', label='Truncamento')
    plt.plot(arred_data['n (casas)'], arred_data['Ea'], 's--', label='Arredondamento')
    
    plt.yscale('log')
    plt.title(f'x = {x}')
    plt.xlabel('n')
    plt.ylabel('Ea')
    plt.legend()
    plt.grid(True, which="both", ls="--", alpha=0.5)

plt.tight_layout()
plt.show()



**Resposta (Parte A, item 5):**
Arredondar nem sempre gera um erro estritamente menor, mas ele é *menor ou igual* ao erro do truncamento. O arredondamento escolhe o valor mais próximo minimizando o erro absoluto, enquanto o truncamento apenas descarta os dígitos. Quando a fração descartada é estritamente menor que a metade da casa decimal de precisão (dígito < 5), truncar e arredondar são matematicamente idênticos, resultando no mesmo erro absoluto. Nos casos onde é maior, arredondar é mais preciso.



## Parte B – Medidas de pressão arterial


In [ ]:
dados = np.array([121.7, 120.4, 122.1, 119.8, 121.2, 120.9, 122.4, 121.0, 120.2, 121.5])
originais = dados
arredondados = np.round(dados)
truncados = np.trunc(dados)
media_ref = np.mean(dados)

series_dict = {
    'Originais': originais,
    'Arredondados': arredondados,
    'Truncados': truncados
}

stats = []
for nome, serie in series_dict.items():
    stats.append({
        'Série': nome,
        'Média': np.mean(serie),
        'Mínimo': np.min(serie),
        'Máximo': np.max(serie),
        'Amplitude': np.max(serie) - np.min(serie),
        'Desvio Padrão (amostral)': np.std(serie, ddof=1)
    })

df_stats = pd.DataFrame(stats)
display(df_stats)

erros_medias = []
for nome in ['Arredondados', 'Truncados']:
    media_aprox = df_stats.loc[df_stats['Série'] == nome, 'Média'].values[0]
    erros_medias.append({
        'Série': nome,
        'Ea': erro_absoluto(media_ref, media_aprox),
        'Er': erro_relativo(media_ref, media_aprox),
        'E%': erro_percentual(media_ref, media_aprox)
    })
df_erros_medias = pd.DataFrame(erros_medias)
display(df_erros_medias)

fig, axs = plt.subplots(2, 2, figsize=(14, 10))

# 1. Gráfico das medições (com a referência como quarta série de base)
axs[0, 0].plot(originais, 'o-', label='Originais')
axs[0, 0].plot(arredondados, 's--', label='Arredondados')
axs[0, 0].plot(truncados, '^-.', label='Truncados')
axs[0, 0].axhline(media_ref, color='r', linestyle=':', label='Média Ref')
axs[0, 0].set_title('Medições')
axs[0, 0].legend()

# 2. Erros absolutos das médias
axs[0, 1].bar(df_erros_medias['Série'], df_erros_medias['Ea'], color=['orange', 'green'])
axs[0, 1].set_title('Ea das Médias')

# 3. Erros percentuais das médias
axs[1, 0].bar(df_erros_medias['Série'], df_erros_medias['E%'], color=['orange', 'green'])
axs[1, 0].set_title('E% das Médias (%)')

# 4. Histograma
axs[1, 1].hist([originais, arredondados], bins=5, label=['Originais', 'Arredondados'])
axs[1, 1].set_title('Histograma')
axs[1, 1].legend()

plt.tight_layout()
plt.show()



**Resposta (Parte B):**
A representação inteira pode mascarar parte da variabilidade inerente da amostra, modificando o desvio padrão e a amplitude originais. Embora a média possa permanecer razoavelmente próxima (especialmente com o arredondamento), sutilezas estatísticas se perdem na discretização, o que significa que essa representação inteira reduz a fidelidade da amostra pequena, tornando-a menos ideal para preservar completamente a variabilidade.



## Parte C – Vazão em um tubo de pequeno diâmetro


In [ ]:
r_ref = 0.80e-3
L = 0.20
mu_ref = 3.5e-3
dp_ref = 1200

def calc_Q(r, dp, mu):
    return (np.pi * r**4 * dp) / (8 * mu * L)

Q_ref = calc_Q(r_ref, dp_ref, mu_ref)

r_aprox = truncar(0.80, 1) * 1e-3
mu_aprox_trunc = truncar(3.5e-3, 3) 
mu_aprox_arred = round(3.5e-3, 3)
dp_aprox_trunc = truncar(1200 / 100, 0) * 100
dp_aprox_arred = round(1200, -2)

Q_r = calc_Q(r_aprox, dp_ref, mu_ref)
Q_mu_trunc = calc_Q(r_ref, dp_ref, mu_aprox_trunc)
Q_mu_arred = calc_Q(r_ref, dp_ref, mu_aprox_arred)
Q_dp = calc_Q(r_ref, dp_aprox_trunc, mu_ref)
Q_all_trunc = calc_Q(r_aprox, dp_aprox_trunc, mu_aprox_trunc)
Q_all_arred = calc_Q(r_aprox, dp_aprox_arred, mu_aprox_arred)

cenarios_C1 = [
    ('Referência', Q_ref),
    ('r trunc/arred', Q_r),
    ('mu trunc (3 casas)', Q_mu_trunc),
    ('mu arred (3 casas)', Q_mu_arred),
    ('dp trunc/arred', Q_dp),
    ('Todas truncadas', Q_all_trunc),
    ('Todas arredondadas', Q_all_arred)
]

res_C1 = [{'Cenário': n, 'Q': q, 'Ea': erro_absoluto(Q_ref, q), 'E%': erro_percentual(Q_ref, q)} for n, q in cenarios_C1]
display(pd.DataFrame(res_C1))

# C.2 Sensibilidade
raios_mm = np.linspace(0.70, 0.90, 50)
Q_ref_list = []
Q_arred_list = []
Q_trunc_list = []
Ea_list, Er_list, Ep_list = [], [], []

for r_mm in raios_mm:
    Q = calc_Q(r_mm * 1e-3, dp_ref, mu_ref)
    
    r_arred = round(r_mm, 1) * 1e-3
    r_trunc = truncar(r_mm, 1) * 1e-3
    
    Q_arred = calc_Q(r_arred, dp_ref, mu_ref)
    Q_trunc = calc_Q(r_trunc, dp_ref, mu_ref)
    
    Q_ref_list.append(Q)
    Q_arred_list.append(Q_arred)
    Q_trunc_list.append(Q_trunc)
    
    Ea_list.append(erro_absoluto(Q, Q_arred)) 
    Er_list.append(erro_relativo(Q, Q_arred))
    Ep_list.append(erro_percentual(Q, Q_arred))

fig, axs = plt.subplots(2, 2, figsize=(14, 10))
axs[0, 0].plot(raios_mm, Q_ref_list, label='Q Ref')
axs[0, 0].plot(raios_mm, Q_arred_list, '--', label='Q r arred')
axs[0, 0].plot(raios_mm, Q_trunc_list, ':', label='Q r trunc')
axs[0, 0].set_title('1. Q vs r')
axs[0, 0].legend()
axs[0, 1].plot(raios_mm, Ea_list, 'g'); axs[0, 1].set_title('2. Ea (r arred)')
axs[1, 0].plot(raios_mm, Er_list, 'orange'); axs[1, 0].set_title('3. Er')
axs[1, 1].plot(raios_mm, Ep_list, 'purple'); axs[1, 1].set_title('4. E%')
plt.tight_layout(); plt.show()



**Explicação (C.2):**
Na equação de Hagen-Poiseuille, a vazão $Q$ é proporcional à quarta potência do raio ($r^4$). Isso significa que a sensibilidade da vazão a pequenas variações no raio é enorme, propagando e amplificando qualquer erro geométrico na medida ou aproximação do raio.



## Parte D – Escoamento em um nanocateter


In [ ]:
r_nano = 50e-9
L_nano = 10e-6
mu_nano = 3.5e-3
dp_nano = 5000

def calc_Q_nano(r):
    return (np.pi * r**4 * dp_nano) / (8 * mu_nano * L_nano)

Q_nano_ref = calc_Q_nano(r_nano)
delta_rs_nm = np.arange(0.1, 5.1, 0.1)

Q_rminus, Q_rplus, Q_refs = [], [], []
ea_maxs, er_maxs, ep_maxs = [], [], []

for dr_nm in delta_rs_nm:
    dr = dr_nm * 1e-9
    Q_m = calc_Q_nano(r_nano - dr)
    Q_p = calc_Q_nano(r_nano + dr)
    
    Q_rminus.append(Q_m)
    Q_rplus.append(Q_p)
    Q_refs.append(Q_nano_ref)
    
    ea_max = max(erro_absoluto(Q_nano_ref, Q_m), erro_absoluto(Q_nano_ref, Q_p))
    ea_maxs.append(ea_max)
    er_maxs.append(ea_max / Q_nano_ref)
    ep_maxs.append((ea_max / Q_nano_ref) * 100)

fig, axs = plt.subplots(2, 2, figsize=(14, 10))
axs[0, 0].plot(delta_rs_nm, Q_refs, label='Ref')
axs[0, 0].plot(delta_rs_nm, Q_rminus, label='r - dr')
axs[0, 0].plot(delta_rs_nm, Q_rplus, label='r + dr')
axs[0, 0].set_title('1. Vazões')
axs[0, 0].legend()
axs[0, 1].plot(delta_rs_nm, ea_maxs, 'g'); axs[0, 1].set_title('2. Ea Máx')
axs[1, 0].plot(delta_rs_nm, er_maxs, 'orange'); axs[1, 0].set_title('3. Er Máx')
axs[1, 1].plot(delta_rs_nm, ep_maxs, 'purple'); axs[1, 1].set_title('4. E% Máx')
plt.tight_layout(); plt.show()

erros_diretos_relativos = np.array(er_maxs)
erros_lineares = 4 * (delta_rs_nm * 1e-9 / r_nano)

plt.figure(figsize=(7,4))
plt.plot(delta_rs_nm, erros_diretos_relativos, label='Direto (Max)')
plt.plot(delta_rs_nm, erros_lineares, '--', label='Aprox Linear (4*dr/r)')
plt.title('Comparação do erro')
plt.legend(); plt.show()



In [ ]:
# D.2 Precisão computacional
for p, np_type in [("float64", np.float64), ("float32", np.float32)]:
    r, dp, mu, L, pi = [np_type(x) for x in [r_nano, dp_nano, mu_nano, L_nano, np.pi]]
    r4 = r**4
    Q = (pi * r4 * dp) / (np_type(8) * mu * L)
    print(f"[{p}] r^4={r4}, Q={Q}, eps={np.finfo(np_type).eps}")

r_64, dp_64, mu_64, L_64, pi_64 = [np.float64(x) for x in [r_nano, dp_nano, mu_nano, L_nano, np.pi]]
Q_64 = (pi_64 * r_64**4 * dp_64) / (np.float64(8) * mu_64 * L_64)

r_32, dp_32, mu_32, L_32, pi_32 = [np.float32(x) for x in [r_nano, dp_nano, mu_nano, L_nano, np.pi]]
Q_32 = (pi_32 * r_32**4 * dp_32) / (np.float32(8) * mu_32 * L_32)

print(f"\nDiferença Absoluta: {np.abs(Q_64 - Q_32)}")
print(f"Diferença Relativa: {np.abs(Q_64 - Q_32) / Q_64}")

print("\nVariações:")
dr_r_vals = [1e-1, 1e-3, 1e-5, 1e-7, 1e-9, 1e-11, 1e-13, 1e-15, 1e-17]
for dr_r in dr_r_vals:
    dr = dr_r * r_64
    q_r_dr = (pi_64 * (r_64 + dr)**4 * dp_64) / (np.float64(8) * mu_64 * L_64)
    delta1 = (q_r_dr - Q_64) / Q_64
    delta2 = (1 + dr_r)**4 - 1
    print(f"dr/r={dr_r:.1e} | delta1={delta1:e} | delta2={delta2:e} | cancelamento={np.abs(delta1-delta2):e}")



**D.3 Fenômeno físico**

Com a drástica redução da escala do tubo, a proporção superfície por volume no interior do escoamento dispara, o que implica que os efeitos viscosos gerados pelo atrito entre o fluido e a parede do nanocateter assumem uma dominância massiva. Um resultado gerado com altíssima precisão de casas decimais em um computador pode ser fisicamente incorreto se o modelo assumido pelo qual o fluido escoa ignorar outros efeitos microscópicos ou quânticos em canais tão ínfimos; ou seja, precisão aritmética de máquina e a validade física de um modelo de meio contínuo (Hagen-Poiseuille) são conceitos intrinsecamente distintos.



## Parte E – Síntese visual e Parte 8 (GIF)


In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 3)

ax0 = fig.add_subplot(gs[0, :]); ax0.axis('off')
ax0.text(0.5, 0.8, 'SÍNTESE VISUAL - ERROS E NANOCATETERES', fontsize=24, ha='center', weight='bold')
ax0.text(0.2, 0.4, f'Pressão Medida (Média): {media_ref:.2f} mmHg\nVazão Macro (Ref): {Q_ref:.2e} m³/s', fontsize=16, bbox=dict(facecolor='lightblue', alpha=0.5, pad=10))
ax0.text(0.7, 0.4, f'Vazão Nano (Ref): {Q_nano_ref:.2e} m³/s\nr_nano = 50 nm', fontsize=16, bbox=dict(facecolor='lightgreen', alpha=0.5, pad=10))

ax1 = fig.add_subplot(gs[1, 0])
trunc_A = df_A[(df_A['Valor (x)'] == x_vals[0]) & (df_A['Tipo']=='Truncamento')]
arred_A = df_A[(df_A['Valor (x)'] == x_vals[0]) & (df_A['Tipo']=='Arredondamento')]
ax1.plot(trunc_A['n (casas)'], trunc_A['Ea'], 'o-', label='Trunc')
ax1.plot(arred_A['n (casas)'], arred_A['Ea'], 's--', label='Arred')
ax1.set_yscale('log'); ax1.set_title('Trunc vs Arred (x1)'); ax1.legend()

ax2 = fig.add_subplot(gs[1, 1])
ax2.plot(delta_rs_nm, ep_maxs, 'm', label='E% Máx Nano')
ax2.set_title('Erros no Nanocateter (E%)')
ax2.legend()

ax3 = fig.add_subplot(gs[1, 2])
ax3.plot(raios_mm, Q_ref_list, 'k')
ax3.set_title('Sensibilidade Q x r (Macro)')

ax4 = fig.add_subplot(gs[2, 0:2]); ax4.axis('off')
ax4.text(0.0, 0.5, f"COMPARAÇÃO f32 vs f64:\nQ_64: {Q_64:e} | Q_32: {Q_32:e}\nErro Relat: {np.abs(Q_64-Q_32)/Q_64:e}\neps64: {np.finfo(np.float64).eps} | eps32: {np.finfo(np.float32).eps}", fontsize=14, family='monospace')

ax5 = fig.add_subplot(gs[2, 2]); ax5.axis('off')
ax5.text(0.0, 0.5, "CONCLUSÃO:\nPequenos deltas radiais dominam\na precisão do modelo devido à quarta potência.\nA precisão computacional (float32 vs 64)\naumenta as discrepâncias em nanoescalas,\ne perdas por cancelamento sublinham a importância\nde formulações algébricas robustas.", fontsize=14)

plt.tight_layout()
plt.savefig('sintese_visual.png', dpi=100)
plt.close()
display(Image('sintese_visual.png'))



In [ ]:
raios_gif = np.linspace(45, 55, 40)
filenames = []

for i, r_nm in enumerate(raios_gif):
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    
    circle = plt.Circle((0, 0), r_nm, color='blue', alpha=0.5)
    ax[0].add_patch(circle)
    ax[0].set_xlim(-60, 60); ax[0].set_ylim(-60, 60); ax[0].set_aspect('equal')
    ax[0].set_title(f"Seção (r = {r_nm:.1f} nm)")
    
    q_atual = calc_Q_nano(r_nm * 1e-9)
    err_pct = erro_percentual(Q_nano_ref, q_atual) if r_nm != 50.0 else 0.0
    ax[0].text(-55, -45, f"Vazão: {q_atual:.2e} m³/s", fontsize=10)
    ax[0].text(-55, -55, f"E% (vs 50nm): {err_pct:.1f}%", fontsize=10)
    
    ax[1].plot(raios_gif[:i+1], [calc_Q_nano(r*1e-9) for r in raios_gif[:i+1]], 'k-', linewidth=2)
    ax[1].plot(r_nm, q_atual, 'ro', markersize=8)
    ax[1].set_xlim(44, 56)
    ax[1].set_ylim(calc_Q_nano(45e-9), calc_Q_nano(55e-9))
    ax[1].set_title("Vazão x Raio"); ax[1].set_xlabel("r (nm)"); ax[1].set_ylabel("Q (m³/s)")
    
    plt.tight_layout()
    fn = f'frame_{i}.png'
    filenames.append(fn)
    plt.savefig(fn)
    plt.close()

try:
    import imageio.v2 as imageio_compat
    imread_func = imageio_compat.imread
except ImportError:
    imread_func = imageio.imread

with imageio.get_writer('nanocateter.gif', mode='I', duration=0.1) as writer:
    for fn in filenames:
        image = imread_func(fn)
        writer.append_data(image)

for fn in filenames:
    os.remove(fn)

display(Image(filename='nanocateter.gif'))

